Clustering Model using K mean and DBscan

This model aims to anwser two questions users may want to find. 

1) KMeans: Which NSW Local Government Areas (LGAs) have similar road crash patterns based on the selected crash features?

2) DBSCAN: Which LGA-year records show unusual crash patterns that do not fit into any identified dense group?


Importing all needed libaries like panda and sklearn

In [1]:
import numpy as np                                   
import pandas as pd                                  
from sklearn.cluster import KMeans, DBSCAN          
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score  

Load the cleaned dataset

In [2]:
data = pd.read_csv("../data/processed/lga_yearly_aggregates_clustering.csv")
print (data.head())

         LGA  Year  Population  severe_count  non_severe_count  Wet-weather %  \
0     Albury  2020       55466            73                50      12.195122   
1   Armidale  2020       29600            47                10      24.561404   
2    Ballina  2020       45663            58                19      22.077922   
3  Balranald  2020        2257             5                 1       0.000000   
4   Bathurst  2020       43444            81                28      23.853211   

   Rural-road %  Crash rate per 100k   Severe %  total_crashes  
0     13.821138           221.757473  59.349593            123  
1     56.140351           192.567568  82.456140             57  
2     23.376623           168.626678  75.324675             77  
3     83.333333           265.839610  83.333333              6  
4     32.110092           250.897707  74.311927            109  


In [3]:
#data = data[data["LGA"] != "Unincorporated NSW"]    # remove Unincorporated data for testing ONLY.
print(data.shape)

(645, 10)


Select the columns for the dataset to be used

In [4]:
Dcolumns = ["Crash rate per 100k", "Severe %", "Wet-weather %", "Rural-road %"]
X = data[Dcolumns].values

Scale the values in the dataset so it counts equally.

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Creating a table to show scores for s which silhouette data

In [6]:
kmeans_results = []

Test K-Means with 2 to 8 clusters and record the silhouette score for each, to help choose K

In [7]:
# s = silhoutte score which then we will choose our K. In CSV kmeans_k_scores.csv  the column will be saved as k instead of s to avoid confusion
for s in range (2,9):
    range_s = KMeans (n_clusters=s, random_state=20)#K-Means with s clusters (random_state keeps the results repeatable)
    s_labels = range_s.fit_predict(X_scaled)# train it on the scaled data 
    silhouette_avg = silhouette_score(X_scaled,s_labels)
    kmeans_results.append({'k': s,'silhouette_score': silhouette_avg})# store S score in the lsit 
    print(f's = {s}  Silhouette Score: {silhouette_avg:.2f}') # Print score 

s = 2  Silhouette Score: 0.30
s = 3  Silhouette Score: 0.31
s = 4  Silhouette Score: 0.28
s = 5  Silhouette Score: 0.29
s = 6  Silhouette Score: 0.29
s = 7  Silhouette Score: 0.25
s = 8  Silhouette Score: 0.27


Train the Kmeans model based on the best/highest silhouette score. 

In [8]:
# The highest score is s = 3 (0.31), so K = 3 is used to create and train the model

K=3
model = KMeans(n_clusters = K , random_state=20)
model.fit(X_scaled)


,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary <random_state>`.",20
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'
Name,Type,Value
"cluster_centers_ cluster_centers_: ndarray of shape (n_clusters, n_features)Coordinates of cluster centers. If the algorithm stops before fullyconverging (see ``tol`` and ``max_iter``), these will not beconsistent with ``labels_``.","ndarray[float64](3, 4)","[

Step 7: Evalute Kmeans using Silhoutte score


In [9]:
#first must give every row a clustering number

predictions = model.predict(X_scaled)

# Now must print one score for the whole result
silhouette_avg = silhouette_score(X_scaled, predictions)
print (f'Silhoutte Score:{silhouette_avg:.2f}')

Silhoutte Score:0.31


DBscans part

Apply DBSCAN

In [10]:
dbscan = DBSCAN(eps=1.0, min_samples=5) # set as maximun distance and set at least 5 samples 
dbscan.fit(X_scaled)#Run dbscans

row = dbscan.labels_ # make sure each row is labeled

Evaluating DBSCAN using silhoutte score

In [11]:
silhouette_avg = silhouette_score(X_scaled, row)
print (f'Silhouette Score: {silhouette_avg:.2f}') # decimals form

Silhouette Score: 0.46


Identify the noise in the dataset

In [12]:
noise = np.sum (row ==-1 ) # showing all rows with negative 1
print(f'Number of noise points: {noise}')


Number of noise points: 41


Testing with diffirent values to detemine which eps is better using noise and silhouette value

In [13]:
for eps_try in [0.5, 1.0, 1.5, 2.0]:                                 # try four eps values
    dbscan_try = DBSCAN(eps=eps_try, min_samples=5)                 
    dbscan_try.fit(X_scaled)                                        
    labels_try = dbscan_try.labels_                                  
    silhouette_avg = silhouette_score(X_scaled, labels_try)          # determining thesilhouette score 
    noise = np.sum(labels_try == -1)                               
    print(f'eps = {eps_try}  Silhouette Score: {silhouette_avg:.2f}  Number of noise points: {noise}')

eps = 0.5  Silhouette Score: -0.21  Number of noise points: 276
eps = 1.0  Silhouette Score: 0.46  Number of noise points: 41
eps = 1.5  Silhouette Score: 0.61  Number of noise points: 11
eps = 2.0  Silhouette Score: 0.64  Number of noise points: 9


Table: Clustering sizes and profiles

In [14]:
from IPython.display import display # for borders 

data["kmean data"] = predictions # get values from kmean
data["dbscans data"] = row # adds a group number
display(data["kmean data"].value_counts()) # making sure no cluster is small or hold everything
cluster_avg = data.groupby("kmean data")[Dcolumns].mean().round(2) # getting the average for the data wihtin each cluster
display(cluster_avg)
display(data[Dcolumns].mean().round(2).to_frame("Average")) # create a another box to display the average of each column based on the clusters
display(data["dbscans data"].value_counts())

kmean data
1    335
0    185
2    125
Name: count, dtype: int64

,Crash rate per 100k,Severe %,Wet-weather %,Rural-road %
kmean data,,,,
0,316.00,76.28,11.35,63.73
1,217.96,67.15,18.42,12.63
2,433.16,67.95,26.93,67.91


,Average
Crash rate per 100k,287.78
Severe %,69.92
Wet-weather %,18.04
Rural-road %,38.00


dbscans data
 0    604
-1     41
Name: count, dtype: int64

Identify which LGAs belong to each K-Means cluster based their crash patterns 

Since our K= 3 we will have 3 clusters:


Cluster 0 : Higher rural-road %, highest casualty share, and highest casualty %

In [15]:
cluster0 = data[data["kmean data"] == 0] # finding hte once in clustering 0                # rows in cluster 0
counts0 = cluster0["LGA"].value_counts()
print("Cluster 0: Rural, highest casualty share %, lowest wet-surface %")                # years each LGA spent in cluster 0
print(counts0[counts0 == 5])    

Cluster 0: Rural, highest casualty share %, lowest wet-surface %
LGA
Cobar        5
Gunnedah     5
Lachlan      5
Lockhart     5
Narrabri     5
Narromine    5
Name: count, dtype: int64


Cluster 1 :  Mostly non-rural areas and low crash rate

In [16]:
cluster1 = data[data["kmean data"] == 1]                 # rows in cluster 1
counts1 = cluster1["LGA"].value_counts()
print("Cluster 1: Mostly non-rural, low crash rate %")                 # years each LGA spent in cluster 1
print(counts1[counts1 == 5])     

Cluster 1: Mostly non-rural, low crash rate %
LGA
Albury                     5
Ballina                    5
Bayside (NSW)              5
Blacktown                  5
Blue Mountains             5
Broken Hill                5
Burwood                    5
Camden                     5
Campbelltown (NSW)         5
Canada Bay                 5
Canterbury-Bankstown       5
Central Coast (NSW)        5
Cessnock                   5
Coffs Harbour              5
Cumberland                 5
Fairfield                  5
Georges River              5
Griffith                   5
Hawkesbury                 5
Hornsby                    5
Hunters Hill               5
Inner West                 5
Kiama                      5
Ku-ring-gai                5
Lake Macquarie             5
Lane Cove                  5
Lismore                    5
Liverpool                  5
Maitland                   5
Mosman                     5
Newcastle                  5
North Sydney               5
Northern Beaches      

Cluster 2: Rural areas, wet-surface share % and highest crash rate

In [17]:
cluster2 = data[data["kmean data"] == 2]       # rows in cluster 2
counts2 = cluster2["LGA"].value_counts()# years each LGA spent in cluster 2
print("Cluster 2: Rural, highest wet-surface %, highest crash rate % ")
print(counts2[counts2 == 5])

Cluster 2: Rural, highest wet-surface %, highest crash rate % 
LGA
Cootamundra-Gundagai    5
Snowy Monaro            5
Snowy Valleys           5
Walcha                  5
Name: count, dtype: int64


Stating the total amount of stable LGAS in each cluster and the total amount of LGAs 

In [18]:
print("Cluster 0 stable LGAs:", len(counts0[counts0 == 5]))     
print("Cluster 1 stable LGAs:", len(counts1[counts1 == 5]))     
print("Cluster 2 stable LGAs:", len(counts2[counts2 == 5]))     
print("Total LGAs:", data["LGA"].nunique())                     

Cluster 0 stable LGAs: 6
Cluster 1 stable LGAs: 54
Cluster 2 stable LGAs: 4
Total LGAs: 129


Which LGAs are unusual? using DBscans 

In [19]:
row_noise = data[data["dbscans data"] == -1] # get all the rows labeled as -1
print (row_noise[["LGA", "Year", "Population", "total_crashes"] + Dcolumns + ["kmean data"]]) 
print(row_noise["LGA"].value_counts())# count the total which is flaged per LGA in a year

                      LGA  Year  Population  total_crashes  \
15             Brewarrina  2020        1525              1   
24             Carrathool  2020        2892             19   
26        Central Darling  2020        1774             19   
44              Gilgandra  2020        4327              9   
50                 Gwydir  2020        5019             16   
71               Lockhart  2020        3308              3   
87                 Oberon  2020        5540             31   
115                Walcha  2020        3057             21   
128    Unincorporated NSW  2020        1055             13   
132             Balranald  2021        2208              7   
140               Blayney  2021        7511             21   
144            Brewarrina  2021        1487              2   
153            Carrathool  2021        2873             20   
161             Coonamble  2021        3833              5   
216                Oberon  2021        5567             37   
237     

Save all csv for visulisation

In [20]:
import os

# CSV of the complete dataset with K-Means and DBSCAN labels
data.to_csv("../data/processed/lga_clusters.csv", index=False)

# CSV to show the average feature values for each K-Means cluster
cluster_avg.to_csv("../data/processed/cluster_profiles.csv")

# CSV that has k mean silhoutte scores
kmeans_scores = pd.DataFrame(kmeans_results)
kmeans_scores.to_csv("../data/processed/kmeans_k_scores.csv", index=False)

# CSV to compare DBSCAN using eps vlaues to determine how the number of clusters and noise points chnage
epsresults =[]

for eps in [0.5, 1.0, 1.5, 2.0]:
    labels = DBSCAN(eps=eps, min_samples = 5).fit_predict(X_scaled)
    clusters = len (set(labels)- {-1})
    noise = sum(labels == -1)

    epsresults.append([eps, clusters, noise]) #place the results into this box

# Save eps results into a eps
pd.DataFrame(epsresults, columns =["eps", "cluster_count", "noise_count"]).to_csv("../data/processed/dbscan_eps_comparison.csv", index=False)

# CSV for all unusal record
data[data["dbscans data"] == -1].to_csv("../data/processed/dbscan_noise_records.csv", index=False)

Save model for prediction in Assignment 3
 Mainly for the prediction sciprt to load and assign new LGA record to a cluster 

 


In [21]:
import joblib

joblib.dump({
    "scaler": scaler,
    "kmeans": model,
    "features": Dcolumns,
    "cluster_names": 
    {
        0: "Rural, highest casualty share, lowest wet-surface %",
        1: "Mostly non-rural, lowest crash rate",
        2: "Rural, highest wet-surface %, highest crash rate",
    },
}, "../models/kmeans_lga.joblib")

print("Model Saved")

Model Saved
